<a href="https://colab.research.google.com/github/helvecioneto/pyfortracc/blob/main/examples/03_Track-Infrared-Dataset/03_Track-Infrared-Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center"; span style="color:#336699"><b><h2>pyForTraCC - Track Infrared (Real Time Data) </h2></b></div>
<hr style="border:2px solid #0077b9;">
<br/>
<div style="text-align: center;font-size: 90%;">
   <sup><a href="https://www.linkedin.com/in/helvecio-leal/"> Helvécio B. Leal Neto, <i class="fab fa-lg fa-orcid" style="color: #a6ce39"></i></a></sup><t>&nbsp;</t>
    <sup><a href="https://www.linkedin.com/in/alan-calheiros-64a252160/">Alan J. P. Calheiros<i class="fab fa-lg fa-orcid" style="color: #a6ce39"></i></a></sup>
   <br/><br/>
    National Institute for Space Research (INPE)
    <br/>
    Avenida dos Astronautas, 1758, Jardim da Granja, São José dos Campos, SP 12227-010, Brazil
    <br/><br/>
    Contact: <a href="mailto:helvecio.neto@inpe.br">helvecio.neto@inpe.br</a>, <a href="mailto:alan.calheiros@inpe.br">alan.calheiros@inpe.br</a>
    <br/><br/>
    Last Update: Nov 6, 2024
</div>

<br/>

<div style="text-align: justify;  margin-left: 25%; margin-right: 25%;">
<b>Abstract.</b> This Jupyter Notebook shows how to use pyfortracc to track latest images from the GOES-16 satellite processed by the CPTEC/INPE.<br>
The algorithm uses the brightness temperature data from the ABI sensor to identify and track precipitating systems over the south America region.<br>
The output data is a tracking table containing the system's lifecyle.
</div>    
<br/>
<div style="text-align: justify;  margin-left: 15%; margin-right: 15%;font-size: 75%; border-style: solid; border-color: #0077b9; border-width: 1px; padding: 5px;">
    <b>In this example, we will use fortracc to compute track of precipitating systems over the globe and explore the output data after the algorithm workflow.
</b>
    <div style="margin-left: 10px; margin-right: 10px; margin-top:10px">
      <p> Leal Neto, H.B.; Calheiros, A.J.P.;  fortracc Algorithm. São José dos Campos, INPE, 2024. <a href="https://github.com/fortracc-project/" target="_blank"> Online </a>. </p>
    </div>
</div>

### Schedule

### Schedule
 [1. Installation](#install)<br>
 [2. Input Data](#input)<br>
 [3. Read Function](#data)<br>
 [4. Parameters (Name_list)](#namelist)<br>
 [5. Tracking Routine](#track)<br>
 [6. Tracking Visualization](#visualization)<br>

<a id='install'></a>
#### 1. Installation

Installing the pyFortraCC package can be done using the pip install command.

All dependencies will be installed in the current Python environment and the code will be ready to use.

In [1]:
# Install latest version of pyfortracc from github
!python -m pip install -q -U git+https://github.com/fortracc/pyfortracc.git@main#egg=pyfortracc

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.2 MB/s eta 0:00:00


<a id='data'></a>
#### 2. Data Input (Download Files)

The get_cptec script downloads the latest processed infrared images from the GOES-16 satellite, provided by CPTEC/INPE (National Institute for Space Research). These images, available for download at this [link](http://ftp.cptec.inpe.br/goes/goes16/retangular/), are from Channel 13, representing infrared data that have been reprojected onto a rectangular grid over South America.

In [61]:
import subprocess
from datetime import datetime, timedelta

# Função para listar arquivos com gsutil
def list_files(prefix, pattern):
    result = subprocess.run(
        ["gsutil", "ls", f"{prefix}{pattern}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    return result.stdout.decode().splitlines()

# Função para gerar o caminho de diretório com base no ano, dia juliano e hora
def get_directory_prefix(bucket, year, julian_day, hour):
    return f"{bucket}/{year}/{julian_day}/{str(hour).zfill(2)}/"

# Função principal
def get_recent_files(bucket, pattern, min_files):
    files = []
    current_time = datetime.utcnow()

    # Loop até encontrar o número mínimo de arquivos
    while len(files) < min_files:
        year = current_time.year
        julian_day = current_time.timetuple().tm_yday  # Obtém o dia juliano
        hour = current_time.hour

        # Gera o prefixo do diretório para a data e hora atuais
        prefix = get_directory_prefix(bucket, year, julian_day, hour)

        found_files = list_files(prefix, pattern)

        # Adiciona arquivos encontrados à lista
        files.extend(found_files)

        # Retrocede uma hora
        current_time -= timedelta(hours=1)

    # Ordena os arquivos por data (não é necessário, pois a busca já é feita de forma decrescente)
    files.sort(reverse=True)

    # Retorna os arquivos mais recentes, conforme o número mínimo solicitado
    return files[:min_files]

# Execução do código com o número de arquivos desejados
bucket = "gs://gcp-public-data-goes-16/ABI-L2-CMIPF"
pattern = "OR_ABI-L2-CMIPF-M6C13_G16*"
min_files = 20  # Número de arquivos a retornar

recent_files = get_recent_files(bucket, pattern, min_files)

In [ ]:
!pip install rioxarray

In [12]:
import subprocess
import os
from datetime import datetime, timedelta
from osgeo import gdal
import shutil
import xarray as xr
import rioxarray
from pyproj import CRS
import concurrent.futures

# Função para listar arquivos com gsutil
def list_files(prefix, pattern):
    result = subprocess.run(
        ["gsutil", "ls", f"{prefix}{pattern}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    return result.stdout.decode().splitlines()

# Função para gerar o caminho de diretório com base no ano, dia juliano e hora
def get_directory_prefix(bucket, year, julian_day, hour):
    return f"{bucket}/{year}/{julian_day}/{str(hour).zfill(2)}/"

# Função para pegar os arquivos mais recentes
def get_recent_files(bucket, pattern, min_files):
    files = []
    current_time = datetime.utcnow()

    # Loop até encontrar o número mínimo de arquivos
    while len(files) < min_files:
        year = current_time.year
        julian_day = current_time.timetuple().tm_yday  # Obtém o dia juliano
        hour = current_time.hour

        # Gera o prefixo do diretório para a data e hora atuais
        prefix = get_directory_prefix(bucket, year, julian_day, hour)

        found_files = list_files(prefix, pattern)

        # Adiciona arquivos encontrados à lista
        files.extend(found_files)

        # Retrocede uma hora
        current_time -= timedelta(hours=1)

    # Ordena os arquivos por data
    files.sort(reverse=True)

    # Retorna os arquivos mais recentes, conforme o número mínimo solicitado
    return files[:min_files]

# Função para reprojetar e recortar a imagem usando GDAL
def crop_reproject_gdal(input_file, output_file, lon_min, lat_min, lon_max, lat_max, output_dir):
    # Copiar o arquivo para o diretório temporário
    temp_file = "/tmp/temp_file.nc"
    subprocess.run(["gsutil", "cp", input_file, temp_file])

    # Abrir o arquivo NetCDF com xarray
    ds = xr.open_dataset(temp_file)

    # Selecionar a variável e a projeção
    var_name = "CMI"  # Nome da variável que você quer processar
    ds = ds[[var_name, "goes_imager_projection"]]

    # Obter a altura do satélite
    sat_height = ds["goes_imager_projection"].attrs["perspective_point_height"]

    # Recalcular as coordenadas
    ds = ds.assign_coords({
        "x": ds["x"].values * sat_height,
        "y": ds["y"].values * sat_height,
    })

    # Definir o CRS a partir da projeção GOES
    crs = CRS.from_cf(ds["goes_imager_projection"].attrs)
    ds = ds.rio.write_crs(crs)

    # Reprojetar para EPSG:4326 (WGS84)
    ds = ds.rio.reproject(dst_crs="EPSG:4326").rename({"x": "lon", "y": "lat"})

    # Aplicar o recorte usando coordenadas de lat/lon
    ds = ds.rio.clip_box(minx=lon_min, miny=lat_min, maxx=lon_max, maxy=lat_max)

    # Salvar o arquivo processado
    ds[var_name].attrs['comments'] = 'Cropped and reprojected to EPSG:4326 by helvecioblneto@gmail.com'

    # Salvar como NetCDF no diretório de saída
    output_path = os.path.join(output_dir, f"{os.path.basename(input_file)}.nc")
    ds.to_netcdf(output_path)

    # Limpeza do arquivo temporário
    os.remove(temp_file)

    # Mover o arquivo processado para o diretório de saída
    print(f'Arquivo reprojetado e recortado salvo em: {output_path}')

# Função que será executada para cada arquivo
def process_file(file, lon_min, lat_min, lon_max, lat_max, output_dir):
    output_file = f"{output_dir}/{os.path.basename(file)}.nc"  # Caminho completo do arquivo de saída
    crop_reproject_gdal(file, output_file, lon_min, lat_min, lon_max, lat_max, output_dir)

# Execução do código com o número de arquivos desejados
bucket = "gs://gcp-public-data-goes-16/ABI-L2-CMIPF"
pattern = "OR_ABI-L2-CMIPF-M6C13_G16*"  # Você pode adaptar para M4C13_G16
min_files = 10  # Número de arquivos a retornar

# Obter os arquivos mais recentes
recent_files = get_recent_files(bucket, pattern, min_files)

# Definir as coordenadas de recorte
lon_min, lat_min = -70, -30  # Coordenadas mínimas (longitude, latitude)
lon_max, lat_max = -50, -10  # Coordenadas máximas (longitude, latitude)

# Definir o diretório de saída
output_dir = 'input/'  # Defina aqui a pasta de saída desejada

# Verificar se o diretório de saída existe, se não, criar
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Usando ThreadPoolExecutor para processar os arquivos em paralelo
with concurrent.futures.ThreadPoolExecutor() as executor:
    # Processar os arquivos em paralelo
    executor.map(process_file, recent_files, [lon_min] * len(recent_files), [lat_min] * len(recent_files),
                 [lon_max] * len(recent_files), [lat_max] * len(recent_files), [output_dir] * len(recent_files))


Arquivo reprojetado e recortado salvo em: output/OR_ABI-L2-CMIPF-M6C13_G16_s20243462100203_e20243462109522_c20243462110005.nc.nc
Arquivo reprojetado e recortado salvo em: output/OR_ABI-L2-CMIPF-M6C13_G16_s20243462110203_e20243462119525_c20243462119599.nc.nc
Arquivo reprojetado e recortado salvo em: output/OR_ABI-L2-CMIPF-M6C13_G16_s20243462020203_e20243462029523_c20243462029593.nc.nc


<a id='namelist'></a>
#### 3. Read Function

The `read_function` function reads the data from the NetCDF file and returns a numpy array with the data.<br>
We select Band 1 of NetCDF file, which corresponds to the infrared channel of the GOES-16 satellite, and divide the data by 100 to convert it to the temperature in Kelvin.

In [13]:
# The function below reads the data from the downloaded files
import xarray as xr
import glob
def read_function(path):
	ds=xr.open_dataset(path)
	return ds['CMI'].data

In [14]:
# Set the lon_min, lon_max, lat_min and lat_max of domain
files = glob.glob('input/*.nc')
ds = xr.open_dataset(files[0])

lon_min = float(ds['lon'].min().values)
lon_max = float(ds['lon'].max().values)
lat_min = float(ds['lat'].min().values)
lat_max = float(ds['lat'].max().values)

<a id='namelist'></a>
#### 4. Parameters: name_list

The `name_list` function creates a list of the files in the directory. The function receives the path to the directory as input and returns a list of the files in the directory.<br>
We track the convective systems by threshold of 235 K and minimum area of 1000 km².

In [15]:
name_list = {} # Set name_list dict
name_list['input_path'] = 'input/'
name_list['output_path'] = 'output/'
name_list['thresholds'] = [235, 210,200]
name_list['min_cluster_size'] = [300, 250,100]
name_list['operator'] = '<='
name_list['timestamp_pattern'] = 'OR_ABI-L2-CMIPF-M6C13_G16_s%Y%j%H%M'
name_list['pattern_position'] = [0,38]
name_list['delta_time'] = 10
name_list['cluster_method'] = 'ndimage'
name_list['lon_min'] = lon_min
name_list['lon_max'] = lon_max
name_list['lat_min'] = lat_min
name_list['lat_max'] = lat_max

# Add correction methods
name_list['spl_correction'] = True # It is used to perform the correction at Splitting events
name_list['mrg_correction'] = True # It is used to perform the correction at Merging events
name_list['inc_correction'] = True # It is used to perform the correction using Inner Core vectors
name_list['opt_correction'] = True # It is used to perform the correction using the Optical Flow method
name_list['elp_correction'] = True # It is used to perform the correction using the Ellipse method
name_list['validation'] = True # It is used to perform the validation of the correction methods

<a id='track'></a>
#### 5. Track Routine

The `track` function receives the data as input and use name_list to track the convective systems.

In [16]:
# Import the library
import pyfortracc

In [17]:
# Track the clusters
pyfortracc.track(name_list, read_function)

Features Extraction:


100%|███████████████████████████████████| 1/1 +                     [Elapsed:00:03 Remaining:<00:00]


Spatial Operations:


100%|███████████████████████████████████| 1/1 +                     [Elapsed:00:00 Remaining:<00:00]


Cluster linking:


100%|███████████████████████████████████| 1/1 +                     [Elapsed:00:00 Remaining:<00:00]


Concatenating:


100%|███████████████████████████████████| 1/1 +                     [Elapsed:00:00 Remaining:<00:00]


<a id='visualize'></a>
#### 6. Visualize the Track Output

`tracking_table` is a pandas DataFrame containing the tracking information of the convective systems.<br>

In [9]:
import duckdb

# Connect to the database
con = duckdb.connect(database=':memory:', read_only=False)

# Read and filter the data from track table
tracking_table = con.execute(f"""SELECT *
                             FROM parquet_scan('output/track/trackingtable/*.parquet',
                             union_by_name=True)
                             """).fetch_df()
# Display the tracking table
display(tracking_table.tail())

,timestamp,uid,iuid,threshold_level,threshold,status,size,lifetime,expansion,min,...,method,u_spl,v_spl,u_mrg,v_mrg,u_opt,v_opt,u_elp,v_elp,cindex
161,2024-12-11 20:20:00,65.0,65.0910,2,200.0,NEW,115,0,NaN,186.654785,...,noc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,162
162,2024-12-11 20:20:00,66.0,66.0354,2,200.0,NEW,195,0,NaN,197.409119,...,noc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,163
163,2024-12-11 20:20:00,67.0,67.0242,2,200.0,NEW,583,0,NaN,188.436951,...,noc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,164
164,2024-12-11 20:20:00,79.0,79.0581,2,200.0,NEW,102,0,NaN,191.755432,...,noc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,165
165,2024-12-11 20:20:00,79.0,79.0303,2,200.0,NEW,174,0,NaN,192.861572,...,noc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,166


The `plot_animation` receives the data and the track as input and plots the data and the track on the same map.<br>
We need to set the dimensions of the plot, the projection, and the extent of the plot.

In [10]:
import numpy as np

# Visualize as animation. (obs: if run in colab, the animation could be fail sometimes, run again to fix)
pyfortracc.plot_animation(read_function=read_function, # Read function
                          figsize=(8,8), # Figure size
                          name_list=name_list, # Name list dictionary
                          start_timestamp = str(tracking_table['timestamp'].min()), # Start timestamp
                          end_timestamp= str(tracking_table['timestamp'].max()), # End timestamp
                          info_col_name=False, # Info column name
                          cbar_title='Temperature(k)', # Colorbar title
                          trajectory=True, # Plot the trajectory
                          smooth_trajectory=True, # Smooth the trajectory
                          cmap='turbo', # Colormap
                          min_val=200, # Min value
                          max_val=235, # Max value
                          nan_value=235, # NaN value
                          nan_operation=np.greater_equal, # NaN operation
                          bound_color='blue', # Bound color
                          info_cols=['uid'], # Info columns from tracking table
                          parallel=False
                          )

Generating animation... 

To zoom in on the region of interest, we can set the extent of the plot to the region of interest.

In [11]:
# Plot the tracking data for a specific region (Sao Paulo state)
sp_lat_min = -25
sp_lat_max = -20
sp_lon_min = -53
sp_lon_max = -45
zoom_region = [sp_lon_min, sp_lon_max, sp_lat_min, sp_lat_max]

# Visualize as animation. (obs: if run in colab, the animation could be fail sometimes, run again to fix)
pyfortracc.plot_animation(read_function=read_function, # Read function
                          figsize=(8,8), # Figure size
                          name_list=name_list, # Name list dictionary
                          start_timestamp = str(tracking_table['timestamp'].min()), # Start timestamp
                          end_timestamp= str(tracking_table['timestamp'].max()), # End timestamp
                          info_cols=['uid','status','lifetime'],
                          cbar_title='Temperature(k)', # Colorbar title
                          trajectory=True, # Plot the trajectory
                          smooth_trajectory=True, # Smooth the trajectory
                          cmap='turbo', # Colormap
                          min_val=200, # Min value
                          max_val=235, # Max value
                          nan_value=235, # NaN value
                          nan_operation=np.greater_equal, # NaN operation
                          zoom_region=zoom_region, # Zoom region
                          bound_color='blue', # Bound color
                          background='google', # Background
                          traj_color='red', # Trajectory color
                          parallel=False
                          )

Generating animation... 